# Notebook 01: Extract Core FIA Tables to Parquet

This notebook extracts the FIA tables needed for the tree-level carbon storage prediction project.

The full FIA DataMart contains many large CSV files. Instead of loading the whole database into memory, this notebook uses DuckDB to extract selected states, selected columns and relevant reference tables. The outputs are saved as Parquet files for faster loading in later notebooks.

This notebook does not clean the data yet. It only creates manageable core extracts for the next stage.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.paths import RAW_DIR, PARQUET_DIR, OUTPUTS_DIR, create_project_dirs
from src.fia_extract import extract_core_tables

create_project_dirs()

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Parquet output folder:", PARQUET_DIR)
print("Outputs folder:", OUTPUTS_DIR)

Project root: e:\BA_DV_Project\COMM074_FIA_Project
Raw data folder: E:\BA_DV_Project\COMM074_FIA_Project\data\raw_entire_fia
Parquet output folder: E:\BA_DV_Project\COMM074_FIA_Project\data\parquet
Outputs folder: E:\BA_DV_Project\COMM074_FIA_Project\outputs


## 1. Check raw FIA files

In [2]:
required_raw_files = [
    "ENTIRE_TREE.csv",
    "ENTIRE_PLOT.csv",
    "ENTIRE_COND.csv",
    "ENTIRE_REF_SPECIES.csv",
    "ENTIRE_REF_SPECIES_GROUP.csv",
    "ENTIRE_REF_FOREST_TYPE.csv",
    "ENTIRE_REF_FOREST_TYPE_GROUP.csv",
    "ENTIRE_COUNTY.csv",
    "ENTIRE_REF_OWNGRPCD.csv"
]

for file in required_raw_files:
    path = RAW_DIR / file
    print(file, "FOUND" if path.exists() else "MISSING")

ENTIRE_TREE.csv FOUND
ENTIRE_PLOT.csv FOUND
ENTIRE_COND.csv FOUND
ENTIRE_REF_SPECIES.csv FOUND
ENTIRE_REF_SPECIES_GROUP.csv FOUND
ENTIRE_REF_FOREST_TYPE.csv FOUND
ENTIRE_REF_FOREST_TYPE_GROUP.csv FOUND
ENTIRE_COUNTY.csv FOUND
ENTIRE_REF_OWNGRPCD.csv FOUND


## 2. Selected states for proof-of-concept

The project uses six selected states: Alabama, California, Georgia, Maine, Oregon and Washington. This provides regional variation while keeping the dataset manageable for the coursework pipeline.

In [3]:
selected_state_codes = [1, 6, 13, 23, 41, 53]

selected_state_codes

[1, 6, 13, 23, 41, 53]

## 3. Extract selected FIA tables

In [4]:
extraction_summary = extract_core_tables(
    selected_state_codes=selected_state_codes
)

extraction_summary

Checking columns in ENTIRE_PLOT.csv
plot_selected: 246,218 rows
Checking columns in ENTIRE_TREE.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

tree_selected: 5,796,627 rows
Checking columns in ENTIRE_COND.csv
cond_selected: 311,972 rows
Checking columns in ENTIRE_REF_SPECIES.csv
ref_species: 2,697 rows
Checking columns in ENTIRE_REF_SPECIES_GROUP.csv
ref_species_group: 54 rows
Checking columns in ENTIRE_REF_FOREST_TYPE.csv
ref_forest_type: 207 rows
Checking columns in ENTIRE_REF_FOREST_TYPE_GROUP.csv
ref_forest_type_group: 34 rows
Checking columns in ENTIRE_COUNTY.csv
county: 3,250 rows
Checking columns in ENTIRE_REF_OWNGRPCD.csv
ref_owngrpcd: 4 rows
Extraction summary saved to E:\BA_DV_Project\COMM074_FIA_Project\outputs\extraction_summary.csv


,table_name,output_file,row_count
0,plot_selected,plot_selected.parquet,246218
1,tree_selected,tree_selected.parquet,5796627
2,cond_selected,cond_selected.parquet,311972
3,ref_species,ref_species.parquet,2697
4,ref_species_group,ref_species_group.parquet,54
5,ref_forest_type,ref_forest_type.parquet,207
6,ref_forest_type_group,ref_forest_type_group.parquet,34
7,county,county.parquet,3250
8,ref_owngrpcd,ref_owngrpcd.parquet,4


In [5]:
tree_selected = pd.read_parquet(PARQUET_DIR / "tree_selected.parquet")

print(tree_selected.columns.tolist())
print("CR included:", "CR" in tree_selected.columns)

['CN', 'PLT_CN', 'PREV_TRE_CN', 'INVYR', 'STATECD', 'UNITCD', 'COUNTYCD', 'PLOT', 'SUBP', 'TREE', 'CONDID', 'STATUSCD', 'SPCD', 'DIA', 'DIAHTCD', 'HT', 'HTCD', 'ACTUALHT', 'CR', 'CCLCD', 'CARBON_AG', 'CARBON_BG', 'DRYBIO_AG', 'DRYBIO_BG', 'VOLCFNET', 'VOLCSNET', 'VOLBFNET', 'TPA_UNADJ']
CR included: True


In [13]:
cond_selected = pd.read_parquet(PARQUET_DIR / "cond_selected.parquet")
print(cond_selected.columns.tolist())
print("STDSZCD included:", "STDSZCD" in cond_selected.columns)
print("OWNGRPCD included:", "OWNGRPCD" in cond_selected.columns)

['CN', 'PLT_CN', 'CONDID', 'COND_STATUS_CD', 'FORTYPCD', 'FLDTYPCD', 'OWNGRPCD', 'RESERVCD', 'SITECLCD', 'STDAGE', 'STDSZCD', 'BALIVE', 'ALSTK', 'GSSTK']
STDSZCD included: True
OWNGRPCD included: True


## 4. Check extracted Parquet files

In [6]:
parquet_files = sorted(PARQUET_DIR.glob("*.parquet"))

for file in parquet_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name}: {size_mb:.2f} MB")

cond_selected.parquet: 7.41 MB
county.parquet: 0.05 MB
plot_selected.parquet: 6.37 MB
ref_forest_type.parquet: 0.00 MB
ref_forest_type_group.parquet: 0.00 MB
ref_owngrpcd.parquet: 0.00 MB
ref_species.parquet: 0.11 MB
ref_species_group.parquet: 0.00 MB
tree_selected.parquet: 278.00 MB


In [7]:
plot_selected = pd.read_parquet(PARQUET_DIR / "plot_selected.parquet")
tree_selected = pd.read_parquet(PARQUET_DIR / "tree_selected.parquet")
cond_selected = pd.read_parquet(PARQUET_DIR / "cond_selected.parquet")

print("Plot shape:", plot_selected.shape)
print("Tree shape:", tree_selected.shape)
print("Condition shape:", cond_selected.shape)

Plot shape: (246218, 13)
Tree shape: (5796627, 28)
Condition shape: (311972, 14)


In [8]:
plot_selected.head()

,CN,STATECD,UNITCD,COUNTYCD,PLOT,PLOT_STATUS_CD,INVYR,MEASYEAR,MEASMON,MEASDAY,LAT,LON,ELEV
0,44540625020004,41,4,45,92194,2.0,2013,2013.0,9.0,1.0,42.223685,-117.701166,5100.0
1,44541690020004,41,4,45,93235,2.0,2013,2013.0,9.0,1.0,43.556785,-117.167603,3300.0
2,44540333020004,41,4,45,93739,2.0,2013,2013.0,9.0,1.0,44.005172,-117.029425,2100.0
3,44543226020004,41,4,45,95160,2.0,2013,2013.0,9.0,1.0,43.360484,-117.168441,4100.0
4,44541685020004,41,4,45,95633,2.0,2013,2013.0,9.0,1.0,42.672177,-117.581170,4100.0


In [9]:
tree_selected.head()

,CN,PLT_CN,PREV_TRE_CN,INVYR,STATECD,UNITCD,COUNTYCD,PLOT,SUBP,TREE,...,CR,CCLCD,CARBON_AG,CARBON_BG,DRYBIO_AG,DRYBIO_BG,VOLCFNET,VOLCSNET,VOLBFNET,TPA_UNADJ
0,157825670010854,157531323010854,NaN,1982,1,5,111,90055,104,3,...,NaN,4.0,5.930856,1.531520,11.861712,3.063039,NaN,0.0,0.0,0.000
1,157825671010854,157531323010854,NaN,1982,1,5,111,90055,104,4,...,NaN,4.0,4.353856,1.151194,8.707712,2.302387,NaN,NaN,NaN,0.000
2,157825672010854,157531323010854,NaN,1982,1,5,111,90055,104,5,...,NaN,4.0,5.930856,1.531520,11.861712,3.063039,NaN,NaN,NaN,0.000
3,157825673010854,157531323010854,NaN,1982,1,5,111,90055,104,6,...,NaN,4.0,3.889810,1.038430,7.779619,2.076860,NaN,NaN,NaN,0.000
4,157825674010854,157531324010854,NaN,1982,1,5,111,90056,101,1,...,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,91.685


In [10]:
cond_selected.head()

,CN,PLT_CN,CONDID,COND_STATUS_CD,FORTYPCD,FLDTYPCD,OWNGRPCD,RESERVCD,SITECLCD,STDAGE,STDSZCD,BALIVE,ALSTK,GSSTK
0,205297170010854,43539075010478,1,1,161.0,161.0,40.0,0.0,5.0,31.0,2.0,113.4604,62.5064,45.9191
1,205297171010854,43539075010478,2,1,520.0,520.0,40.0,0.0,4.0,30.0,2.0,38.5134,28.0173,26.6401
2,205297185010854,43540262010478,1,1,520.0,406.0,40.0,0.0,4.0,25.0,2.0,48.9767,36.1633,18.1259
3,205297182010854,43540098010478,1,1,161.0,161.0,40.0,0.0,5.0,31.0,1.0,63.2295,39.5035,39.0869
4,205297183010854,43540098010478,2,1,404.0,406.0,40.0,0.0,5.0,15.0,2.0,66.5839,36.9382,26.3756


In [11]:
plot_selected["STATECD"].value_counts().sort_index()

STATECD
1     42677
6     43814
13    73796
23    21638
41    41194
53    23099
Name: count, dtype: int64

In [12]:
print("TREE columns:")
print(tree_selected.columns.tolist())

print("\nPLOT columns:")
print(plot_selected.columns.tolist())

print("\nCOND columns:")
print(cond_selected.columns.tolist())

TREE columns:
['CN', 'PLT_CN', 'PREV_TRE_CN', 'INVYR', 'STATECD', 'UNITCD', 'COUNTYCD', 'PLOT', 'SUBP', 'TREE', 'CONDID', 'STATUSCD', 'SPCD', 'DIA', 'DIAHTCD', 'HT', 'HTCD', 'ACTUALHT', 'CR', 'CCLCD', 'CARBON_AG', 'CARBON_BG', 'DRYBIO_AG', 'DRYBIO_BG', 'VOLCFNET', 'VOLCSNET', 'VOLBFNET', 'TPA_UNADJ']

PLOT columns:
['CN', 'STATECD', 'UNITCD', 'COUNTYCD', 'PLOT', 'PLOT_STATUS_CD', 'INVYR', 'MEASYEAR', 'MEASMON', 'MEASDAY', 'LAT', 'LON', 'ELEV']

COND columns:
['CN', 'PLT_CN', 'CONDID', 'COND_STATUS_CD', 'FORTYPCD', 'FLDTYPCD', 'OWNGRPCD', 'RESERVCD', 'SITECLCD', 'STDAGE', 'STDSZCD', 'BALIVE', 'ALSTK', 'GSSTK']


In [14]:
required_tree_cols = ["PLT_CN", "CONDID", "SPCD", "DIA", "HT", "ACTUALHT", "CR", "CARBON_AG"]
required_cond_cols = ["PLT_CN", "CONDID", "FORTYPCD", "OWNGRPCD", "SITECLCD", "STDAGE", "STDSZCD"]
required_plot_cols = ["CN", "STATECD", "COUNTYCD", "INVYR", "MEASYEAR"]

print("Missing TREE columns:", [c for c in required_tree_cols if c not in tree_selected.columns])
print("Missing COND columns:", [c for c in required_cond_cols if c not in cond_selected.columns])
print("Missing PLOT columns:", [c for c in required_plot_cols if c not in plot_selected.columns])

Missing TREE columns: []
Missing COND columns: []
Missing PLOT columns: []


In [15]:
key_tree_cols = [c for c in required_tree_cols if c in tree_selected.columns]

tree_selected[key_tree_cols].isna().mean().mul(100).round(2).sort_values(ascending=False)

ACTUALHT     31.18
CR           29.10
HT           18.25
CARBON_AG    12.75
DIA           5.03
SPCD          0.00
PLT_CN        0.00
CONDID        0.00
dtype: float64

## Summary

This notebook extracted the core FIA tables required for the tree-level carbon storage prediction project. The extraction used selected states and selected columns to reduce the size of the raw FIA DataMart while keeping the fields needed for later cleaning, EDA and modelling.

The extracted TREE table includes key tree-level variables such as `DIA`, `HT`, `ACTUALHT`, `CR`, `SPCD` and `CARBON_AG`. The extracted COND table includes site and stand variables such as `FORTYPCD`, `OWNGRPCD`, `SITECLCD`, `STDAGE` and `STDSZCD`. The extracted PLOT table includes geographic and inventory context such as `STATECD`, `COUNTYCD` and `INVYR`.

The outputs of this notebook are Parquet files saved in `data/parquet/`. These files will be used in the next notebook to merge TREE, PLOT and COND into one tree-level dataset.